# Week 13 Lab — Stresses that change with time

**HWRS 564a · Fall 2026**

Every model so far has been **steady state**: the answer is what the aquifer
settles down to if the stresses never change. That is a useful question and it is
not the one water managers ask.

They ask *when*. When does the water table reach the pump intake? How long until
the cone reaches the county line? How much of what we pump this year comes out of
storage rather than out of recharge?

Answering those needs a **transient** model, which means time steps, storage
properties, and stresses that vary — and one new package, `EVT`, for
evapotranspiration.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Set up multiple stress periods with `perlen`, `nstp`, and `steady`
2. Explain what `sy` and `ss` are and when each one applies
3. Vary a well's rate between stress periods
4. Read a time series of heads out of the `.hds` file
5. Use the budget to say how much pumped water came out of storage
6. Add `EVT` and explain what the extinction depth does

---

## Part 1 — Setup

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
MF_EXE = ROOT / "modflow" / "mf2005"
assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Run ./postbuild.sh from a terminal."
)

RUN = ROOT / "_run"
land_surface = np.loadtxt(DATA / "tucson_grid_top.csv", delimiter=",")

NROW, NCOL, DELR = 40, 60, 250.0
AQ_TOP, AQ_BOT = 600.0, 400.0
WELL_ROW, WELL_COL = 20, 30

UA_RED, UA_BLUE, UA_SKY = "#AB0520", "#0C234B", "#81D3EB"
print("ready")

---

## Part 2 — Time in MODFLOW

MODFLOW splits a simulation into **stress periods**, and each stress period into
**time steps**.

- A **stress period** is an interval over which every stress is constant. Change
  the pumping rate and you need a new stress period.
- A **time step** is a solver interval within a stress period. More steps means a
  more accurate transient response and a slower run.

Three arrays in `DIS` control it:

| Argument | Meaning |
|---|---|
| `perlen` | length of each stress period, in the model's time units |
| `nstp` | number of time steps within each stress period |
| `steady` | one flag per period: steady state or transient |

In [ ]:
# steady state, then a 30-day period with DAILY steps, then ten annual periods
perlen = [1.0, 30.0] + [365.0] * 10
nstp = [1, 30] + [4] * 10
steady = [True] + [False] * 11
NPER = len(perlen)

total_days = sum(p for p, s in zip(perlen, steady) if not s)
print(f"{NPER} stress periods, {sum(nstp)} time steps")
print(f"transient simulation covers {total_days:,.0f} days = {total_days / 365:.1f} years")

> **Why the odd 30-day period with daily steps?** Because the interesting part of
> a transient response happens at the start, and a quarterly time step steps
> straight over it. Fine early steps cost almost nothing — the whole run is a few
> seconds — and without them Part 5 of this notebook has nothing to show.
>
> Choosing time steps is like choosing cell size: **resolve the thing you came to
> look at.**

> **Why start with a steady-state period?** A transient model needs an initial
> head field, and any field you invent will be out of equilibrium with the
> boundaries — so the model spends its first years relaxing towards balance and
> you cannot tell that artificial drift apart from the response you are trying to
> measure.
>
> Running one steady-state period first, with no pumping, produces heads that are
> *already* in equilibrium. Everything after that is the response to the stress
> you added. **This is standard practice and skipping it is a common error.**

### YOUR TURN 1

Confirm the time discretization is what you think it is.

- `n_transient_periods` — how many periods are transient
- `days_per_annual_step` — length of one time step in the *annual* periods
- `total_output_times` — how many head snapshots a run will produce, if you save
  every time step

In [ ]:
# YOUR TURN
n_transient_periods = ...
days_per_annual_step = ...
total_output_times = ...

In [ ]:
# CHECK
assert n_transient_periods == 11, f"got {n_transient_periods}"
assert abs(days_per_annual_step - 91.25) < 0.01, f"got {days_per_annual_step}"
assert total_output_times == 71, f"got {total_output_times}"
print(f"{n_transient_periods} transient periods")
print(f"{days_per_annual_step:.2f} days per step in the annual periods")
print(f"{total_output_times} head snapshots")
print("Correct.")

---

## Part 3 — Storage: `sy` and `ss`

A steady-state model needs no storage properties — nothing is changing. A
transient one does, because water has to come from somewhere while heads fall.

| Property | Symbol | Applies when | Typical value | Units |
|---|---|---|---|---|
| **Specific yield** | `sy` | unconfined — pores actually drain | 0.05 – 0.30 | dimensionless |
| **Specific storage** | `ss` | confined — water expands, matrix compresses | 1e-6 – 1e-4 | 1/m |

**Those differ by three to four orders of magnitude.** A confined aquifer
releases almost no water per metre of head decline, which is why confined
drawdown propagates so fast and so far.

In [ ]:
SY, SS = 0.15, 1e-5
cell_area = DELR * DELR
thickness = AQ_TOP - AQ_BOT

unconfined_release = SY * cell_area * 1.0
confined_release = SS * cell_area * thickness * 1.0

print(f"water released by 1 m of decline in one {DELR:.0f} m cell:")
print(f"  unconfined (sy = {SY})      {unconfined_release:10,.1f} m3")
print(f"  confined   (ss = {SS:.0e})  {confined_release:10,.1f} m3")
print(f"  ratio                      {unconfined_release / confined_release:10,.0f}x")

### YOUR TURN 2

Our model is confined (`laytyp=0`), so `ss` is what governs it. Work out how much
water the whole aquifer can supply from storage before the answer has to come
from somewhere else.

- `storage_per_metre` — water released by 1 m of head decline across all 2,400 cells
- `days_of_pumping` — how many days of 20,000 m³/d that would cover

In [ ]:
PUMP_RATE = 20000.0     # m3/d

# YOUR TURN
storage_per_metre = ...
days_of_pumping = ...

In [ ]:
# CHECK
assert abs(storage_per_metre - 300_000.0) < 10.0, f"got {storage_per_metre}"
assert abs(days_of_pumping - 15.0) < 0.5, f"got {days_of_pumping}"
print(f"storage released per metre of decline: {storage_per_metre:,.0f} m3")
print(f"that covers {days_of_pumping:.1f} days of pumping at {PUMP_RATE:,.0f} m3/d")
print("Correct.")

**About two weeks, as an upper bound.** Even if the entire aquifer dropped a
metre, storage could only cover a fortnight of this well. That is why a confined
aquifer reaches something close to steady state so quickly: there is very little
in storage, so the cone has to keep expanding until it intercepts enough recharge
or boundary flow to balance the well.

Run the same numbers unconfined and storage per metre goes up by a factor of
`sy / (ss * b)` = `0.15 / (1e-5 x 200)` = **75**. That is over three years of
pumping available from storage rather than two weeks. **Same well, same
conductivity, completely different timescale** — set by the storage property
alone.

---

## Part 4 — A transient run

The well switches on at the start of year 1 and stays on.

In [ ]:
WS = RUN / "week13_transient"
WS.mkdir(parents=True, exist_ok=True)

mf = flopy.modflow.Modflow("trans", model_ws=str(WS), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(
    mf, 1, NROW, NCOL, delr=DELR, delc=DELR,
    top=AQ_TOP, botm=AQ_BOT,
    nper=NPER, perlen=perlen, nstp=nstp, steady=steady,
)

ibound = np.ones((1, NROW, NCOL), dtype=int)
ibound[:, :, 0] = -1
ibound[:, :, -1] = -1
strt = np.full((1, NROW, NCOL), 660.0)
strt[:, :, 0] = 620.0
strt[:, :, -1] = 700.0
flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

# storage properties matter now
flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=0, sy=SY, ss=SS, ipakcb=53)

# stress period 0 is the no-pumping steady state; the well runs from period 1
wel_spd = {p: [[0, WELL_ROW, WELL_COL, 0.0 if p == 0 else -PUMP_RATE]]
           for p in range(NPER)}   # period 0 is the no-pumping steady state
flopy.modflow.ModflowWel(mf, stress_period_data=wel_spd, ipakcb=53)
flopy.modflow.ModflowRch(mf, rech={p: 1.2e-4 for p in range(NPER)}, ipakcb=53)

flopy.modflow.ModflowPcg(mf)
flopy.modflow.ModflowOc(
    mf,
    stress_period_data={(p, s): ["save head", "save budget"]
                        for p in range(NPER) for s in range(nstp[p])},
)

mf.write_input()
success, buff = mf.run_model(silent=True, report=True)
assert success, "MODFLOW did not converge:\n" + "\n".join(buff[-20:])
print("converged")

Note the `stress_period_data` dictionary: **one entry per stress period**, keyed
by period number. That is how any stress varies in time — the well package, the
recharge package, all of them take the same shape.

In [ ]:
hds = flopy.utils.HeadFile(str(WS / "trans.hds"))
times = np.array(hds.get_times())

print(f"{len(times)} output times, from {times[0]:.1f} to {times[-1]:,.1f} days")
print(f"first few: {np.round(times[:5], 1)}")

### YOUR TURN 3

Pull the head history at the wellfield out of the file.

- `well_history` — head in the well cell at every output time, as a 1D array
- `years` — the output times converted to years since pumping began
  (the steady-state period occupies day 0–1, so subtract 1 day then divide by 365)
- `total_drawdown` — head at the first output time minus head at the last

In [ ]:
# YOUR TURN
well_history = ...
years = ...
total_drawdown = ...

In [ ]:
# CHECK
assert well_history.shape == times.shape, f"{well_history.shape} vs {times.shape}"
assert abs(well_history[0] - 662.04) < 0.2, f"first value {well_history[0]}"
assert abs(total_drawdown - 7.67) < 0.2, f"got {total_drawdown}"
assert abs(years[-1] - 10.08) < 0.05, f"last time should be ~10.1 years, got {years[-1]}"
print(f"head before pumping   {well_history[0]:8.2f} m")
print(f"head after 10 years   {well_history[-1]:8.2f} m")
print(f"total drawdown        {total_drawdown:8.2f} m")
print("Correct.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))

axes[0].plot(years, well_history, "o-", color=UA_RED, ms=4)
axes[0].set_xlabel("years since pumping began")
axes[0].set_ylabel("head in the wellfield (m)")
axes[0].set_title("Most of the drawdown happens immediately")
axes[0].grid(alpha=0.3)

# how much of the final drawdown is reached by each time
fraction = (well_history[0] - well_history) / total_drawdown * 100
axes[1].plot(years, fraction, "s-", color=UA_BLUE, ms=4)
axes[1].axhline(90, color="grey", ls=":", lw=1)
axes[1].text(4, 91, "90% of final drawdown", fontsize=9, color="grey")
axes[1].set_xlabel("years since pumping began")
axes[1].set_ylabel("% of final drawdown reached")
axes[1].set_title("Approach to the steady-state answer")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### YOUR TURN 4

The right panel says the model reaches most of its final drawdown quickly. Put a
number on it.

- `pct_after_one_year` — percentage of the final drawdown reached after 1 year
- `years_to_90pct` — the first time in `years` at which 90% is reached

`np.argmax` on a boolean array gives the index of the first `True`.

In [ ]:
# YOUR TURN
pct_after_one_year = ...
years_to_90pct = ...

In [ ]:
# CHECK
assert pct_after_one_year > 80, (
    f"a confined aquifer should get most of the way in one year: got {pct_after_one_year:.0f}%"
)
assert years_to_90pct < 3.0, f"got {years_to_90pct}"
print(f"after 1 year:  {pct_after_one_year:.1f}% of the final drawdown")
print(f"90% reached at {years_to_90pct:.2f} years")
print("Correct.")

**A confined aquifer is nearly at steady state within a year or two.** So for
this system, the steady-state model from Week 11 was answering a question about
the near future, not the distant one — which is worth knowing, because it means
the transient run added detail rather than changing the conclusion.

That would not be true of an unconfined aquifer with `sy = 0.15`. There the
approach is slow enough that a steady-state model describes a condition decades
away, and a manager asking about the next ten years needs the transient answer.

---

## Part 5 — Where transient water comes from

In a steady-state model the budget balances recharge against pumping. In a
transient one there is a third source: **storage**.

In [ ]:
def budget_by_period(list_path):
    """Storage, recharge, wells and constant-head terms for each budget block."""
    lines = Path(list_path).read_text().splitlines()
    starts = [i for i, l in enumerate(lines) if "VOLUMETRIC BUDGET" in l]

    records = []
    for s in starts:
        block = lines[s:s + 30]
        half, terms = "in", {}
        for line in block:
            t = line.strip()
            if t.startswith("OUT:"):
                half = "out"
            if "=" not in t or t.startswith(("TOTAL", "IN - OUT", "PERCENT")):
                continue
            name = t.split("=")[0].strip()
            terms.setdefault(name, {})[half] = float(t.split()[-1])
        records.append(terms)
    return records


blocks = budget_by_period(WS / "trans.list")
print(f"{len(blocks)} budget blocks written")

final = blocks[-1]
for name, v in final.items():
    print(f"  {name:16s} in {v.get('in', 0):11,.1f}   out {v.get('out', 0):11,.1f}")

MODFLOW writes **one budget block per stress period**, at the end of it — not one
per time step. So `blocks[1]` is the end of the 30-day period and `blocks[-1]` is
the end of year 10.

### YOUR TURN 5

Compare the end of the first month of pumping with the end of year ten, and show
where the water came from in each.

- `storage_early` — net storage contribution in `blocks[1]` (in minus out)
- `storage_late` — the same for `blocks[-1]`
- `storage_share_early` — storage as a percentage of the pumping rate, early

In [ ]:
# YOUR TURN
storage_early = ...
storage_late = ...
storage_share_early = ...

In [ ]:
# CHECK
assert storage_early > storage_late, (
    "storage should contribute more early on than after ten years"
)
assert storage_share_early > 10.0, (
    f"storage should still be a big share after one month: got {storage_share_early:.2f}%"
)
assert storage_share_early < 100.0
print(f"net storage, end of the first month : {storage_early:10,.1f} m3/d")
print(f"net storage, end of year 10         : {storage_late:10,.1f} m3/d")
print(f"storage as a share of pumping, early: {storage_share_early:9.1f} %")
print("Correct.")

**The source of pumped water changes over time**, and that is the single most
policy-relevant output of a transient model.

After one month of pumping, **about a quarter of the well's yield is water taken
out of storage** — a one-off withdrawal from the aquifer's stock, not a renewable
supply. By year ten storage contributes essentially nothing, and the entire yield
is **captured**: recharge that would have gone somewhere else, or flow diverted
from the boundaries.

That distinction — *depletion* versus *capture* — is the whole basis of safe-yield
regulation, and you can only see it in the budget.

---

## Part 6 — Evapotranspiration

Where the water table is shallow, plants and soil reach it. `EVT` removes water
at a rate that depends on how deep the water table is.

Three arguments:

| Argument | Meaning |
|---|---|
| `surf` | the elevation at which ET is at its maximum rate |
| `evtr` | that maximum rate, as a flux (m/d) |
| `exdp` | **extinction depth** — how far below `surf` ET reaches zero |

Between `surf` and `surf - exdp`, MODFLOW interpolates linearly. Below it, ET
stops.

In [ ]:
WS_ET = RUN / "week13_et"
WS_ET.mkdir(parents=True, exist_ok=True)

# A shallow-water-table version, so ET has something to work on
mf_et = flopy.modflow.Modflow("et", model_ws=str(WS_ET), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(mf_et, 1, NROW, NCOL, delr=DELR, delc=DELR,
                         top=700.0, botm=400.0, nper=1, steady=True)
ib = np.ones((1, NROW, NCOL), dtype=int)
ib[:, :, 0] = -1
ib[:, :, -1] = -1
strt_et = np.full((1, NROW, NCOL), 690.0)
strt_et[:, :, 0] = 685.0
strt_et[:, :, -1] = 695.0
flopy.modflow.ModflowBas(mf_et, ibound=ib, strt=strt_et)
flopy.modflow.ModflowLpf(mf_et, hk=12.0, laytyp=0, ipakcb=53)
flopy.modflow.ModflowRch(mf_et, rech=6.0e-4, ipakcb=53)

ET_SURF = 700.0        # ET is at its maximum when head reaches 700 m
ET_RATE = 3.0e-3       # m/d, about 1100 mm/yr — riparian vegetation
ET_EXDP = 8.0          # m, extinction depth

flopy.modflow.ModflowEvt(mf_et, surf=ET_SURF, evtr=ET_RATE, exdp=ET_EXDP,
                         nevtop=1, ipakcb=53)
flopy.modflow.ModflowPcg(mf_et)
flopy.modflow.ModflowOc(mf_et, stress_period_data={(0, 0): ["save head", "save budget"]})
mf_et.write_input()
ok_et, buff_et = mf_et.run_model(silent=True, report=True)
assert ok_et, "\n".join(buff_et[-15:])

head_et = flopy.utils.HeadFile(str(WS_ET / "et.hds")).get_data()
print(f"head {head_et.min():.2f} to {head_et.max():.2f} m")
print(f"depth below the ET surface: {ET_SURF - head_et.max():.2f} to "
      f"{ET_SURF - head_et.min():.2f} m")

### YOUR TURN 6

Work out how much ET the model *could* apply at maximum rate, and compare with
what it actually removed.

- `max_possible_et` — `ET_RATE` times the area of all active cells
- `actual_et` — the `ET` term from the budget (out side)
- `et_fraction` — actual as a percentage of maximum

In [ ]:
et_terms = budget_by_period(WS_ET / "et.list")[-1]
print("budget terms:", sorted(et_terms))

# YOUR TURN
max_possible_et = ...
actual_et = ...
et_fraction = ...

In [ ]:
# CHECK
assert actual_et > 0, "the ET package should be removing water"
assert 0 < et_fraction < 100, (
    f"ET should be somewhere between zero and the maximum: got {et_fraction:.1f}%"
)
print(f"maximum possible ET  {max_possible_et:10,.0f} m3/d")
print(f"actual ET            {actual_et:10,.0f} m3/d")
print(f"fraction of maximum  {et_fraction:9.1f} %")
print("Correct.")

ET comes in below the maximum because the water table sits partway down the
extinction interval — so MODFLOW applies a scaled rate, cell by cell.

**Why the extinction depth matters more than it looks.** It creates a feedback:
pumping lowers the water table, which reduces ET, which leaves more water in the
aquifer. So a fraction of what a well produces is not new water at all — it is
water that would otherwise have been transpired by the vegetation along the
river.

That is not an accounting curiosity. It is the mechanism behind riparian habitat
loss along the Santa Cruz, and a model without an `EVT` package cannot represent
it at all.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

- **HW 11 — 2D steady-state model**, Wednesday 11/25 at 11:59pm
- **Project 3 analysis, part 1**

## Next week

Layered models, rivers, and loading a model somebody else built. One session
only — Thanksgiving is Thursday.

## Stuck?

- `stress_period_data` must have an entry for **every** stress period, or FloPy
  reuses the previous one silently. Build it with a dict comprehension.
- A transient model that runs but shows no change usually has `steady=True` in
  every period — check the `DIS` flags.
- `KeyError` on a budget term means the package didn't write to the budget file:
  add `ipakcb=53`.
- If the heads barely move over ten years, check `ss` — a value of 1e-7 makes the
  aquifer respond almost instantly.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.